![Logo UNAL CHEC](https://www.funcionpublica.gov.co/documents/d/guest/logo-universidad-nacional)



# **Agentes**
### Departamento de Ingeniería Eléctrica, Electrónica y Computación
#### Universidad Nacional de Colombia - Sede Manizales

#### Profesor: Diego A. Pérez

Introducción general
Se presentará una introducción práctica al uso de LangChain con modelos vía API de lenguaje Natural como  GPT y Gemini, centrada en cuatro escenarios:
1. Un agente sencillo (generación y análisis tabular).
2. Un sistema RAG básico (búsqueda y recuperación de contexto).
3. Un agente RAG con razonamiento autónomo (Agentic RAG).
4. Un Agentic RAG con herramientas externas.


In [8]:
# Dependencias para Google Colab (Python 3.12 y 3.13).
# Se conserva LangChain 0.3.x para mantener los imports del taller.
# Evita forzar NumPy, Pandas, Requests y OpenTelemetry: Colab los administra.
# Ejecuta esta celda UNA VEZ y reinicia el entorno antes de continuar.

%pip install -q --upgrade-strategy only-if-needed \
    "langchain==0.3.27" \
    "langchain-core==0.3.80" \
    "langchain-community==0.3.27" \
    "langchain-openai==0.3.28" \
    "langchain-google-genai==2.1.8" \
    "langchain-chroma==0.2.5" \
    "langchain-experimental==0.3.4" \
    "langchain-text-splitters==0.3.9" \
    "langchain-ollama==0.3.6" \
    "chromadb>=1.0.9,<2.0.0" \
    "pypdf>=5.0.0,<7.0.0" \
    "tabulate>=0.9.0" \
    "numpy" "pandas" "matplotlib" "colab-xterm"


**Antes de seguir:** tras la instalación, en Google Colab selecciona **Entorno de ejecución → Reiniciar entorno de ejecución**. Luego continúa con la verificación y las demás celdas (si reinicias, las bibliotecas continúan instaladas). La descarga de modelos y el servicio **Ollama** requieren pasos adicionales en la sección *Implementación local*.

Para el RAG debes subir los archivos PDF indicados en `PDF_PATH`, `PDF_PATH_1` y `PDF_PATH_2`. Las APIs externas requieren sus claves en **Colab Secrets**. No compartas el cuaderno con claves en texto plano.

In [9]:
# Verificación local sin gastar créditos de las API.
from importlib import import_module
from importlib.metadata import version
import subprocess
import sys

paquetes = {
    "langchain": "langchain",
    "langchain-core": "langchain_core",
    "langchain-community": "langchain_community",
    "langchain-openai": "langchain_openai",
    "langchain-google-genai": "langchain_google_genai",
    "langchain-chroma": "langchain_chroma",
    "langchain-experimental": "langchain_experimental",
    "langchain-text-splitters": "langchain_text_splitters",
    "langchain-ollama": "langchain_ollama",
    "chromadb": "chromadb",
    "pypdf": "pypdf",
    "tabulate": "tabulate",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "colab-xterm": "colabxterm",
}

print("Python:", sys.version.split()[0])
errores = []
for paquete, modulo in paquetes.items():
    try:
        import_module(modulo)
        print(f"OK  {paquete}=={version(paquete)}")
    except Exception as e:
        errores.append((paquete, str(e)))
        print(f"ERROR  {paquete}: {e}")

# Comprobamos los objetos usados en el cuaderno (sin llamar a las APIs).
try:
    from langchain_community.document_loaders import PyPDFLoader
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_openai import ChatOpenAI, OpenAIEmbeddings
    from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
    from langchain_chroma import Chroma
    from langchain.chains import create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain
    from langchain.agents import AgentExecutor, create_tool_calling_agent
    from langchain.tools import Tool
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_experimental.agents import create_pandas_dataframe_agent
    from langchain_ollama import ChatOllama
    print("OK  Todos los imports de LangChain del taller se resolvieron.")
except Exception as e:
    errores.append(("imports de LangChain", str(e)))
    print("ERROR  Importación del taller:", e)

estado = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
print("\npip check:", "sin conflictos declarados" if estado.returncode == 0 else "advierte conflictos")
if estado.returncode:
    print("\n".join(estado.stdout.splitlines()[:20]))

if errores:
    raise RuntimeError("Hay dependencias/importaciones sin resolver; revisa las líneas ERROR.")


Python: 3.13.18
OK  langchain==0.3.27
OK  langchain-core==0.3.80
OK  langchain-community==0.3.27
OK  langchain-openai==0.3.28
OK  langchain-google-genai==2.1.8
OK  langchain-chroma==0.2.5
OK  langchain-experimental==0.3.4
OK  langchain-text-splitters==0.3.9
OK  langchain-ollama==0.3.6
OK  chromadb==1.5.9
OK  pypdf==6.21.4
OK  tabulate==0.9.0
OK  pandas==2.2.3
OK  numpy==2.4.6
OK  matplotlib==3.10.0
OK  colab-xterm==0.2.0
OK  Todos los imports de LangChain del taller se resolvieron.

pip check: advierte conflictos
ipython 7.34.0 requires jedi, which is not installed.
google-adk 2.7.1 has requirement opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2.
google-adk 2.7.1 has requirement opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2.
langgraph-prebuilt 1.1.1 has requirement langchain-core>=1.3.1, but you have langchain-core 0.3.80.
langgraph 1.2.19 has requirement langchain-core<2,>=1.4.7, but you have langchain-core 0.3.80.
tweepy 4.17.0 has re

### Embeddings internos vs. Embeddings externos: la clave entre un Agente clásico y un RAG

Los **modelos de lenguaje grandes (LLMs)** como *Gemini*, *GPT* o *LLaMA* convierten el texto en números antes de procesarlo.  
Sin embargo, existen **dos formas distintas** de representar y usar esas conversiones numéricas (*embeddings*), y esta diferencia marca la frontera entre un **agente clásico** y un **sistema RAG**.

---

#### Embeddings internos — dentro del modelo

- Son parte de la **arquitectura interna del LLM**.  
- Se generan cada vez que el modelo procesa texto.  
- No se guardan ni se reutilizan fuera del contexto de una conversación.  
- Su propósito es **comprender y generar lenguaje**, no almacenar información.

Ejemplo:
> Cuando el modelo recibe una pregunta, convierte cada palabra en vectores temporales para razonar sobre ella, pero olvida esos vectores al terminar la respuesta.

---

#### Embeddings externos — fuera del modelo (RAG)

- Se obtienen con un **modelo especializado de embeddings** (por ejemplo, `text-embedding-004` o `GoogleGenerativeAIEmbeddings`).  
- Se usan para **convertir documentos** (PDFs, normas, reportes, artículos) en vectores fijos y persistentes.  
- Estos vectores se almacenan en una **base vectorial** (FAISS, Chroma, Milvus, etc.).  
- Permiten **buscar información semánticamente similar**, incluso si las palabras no coinciden literalmente.

Ejemplo:
> El sistema convierte cada párrafo del *RETIE* en un vector y lo guarda.  
> Luego, cuando se pregunta *“¿Qué establece el RETIE sobre la puesta a tierra?”*, el sistema busca los fragmentos más cercanos en el espacio vectorial y se los entrega al modelo para responder con fundamento.

---

#### Diferencia esencial

| Aspecto | Agente clásico | RAG |
|----------|----------------|-----|
| Fuente de conocimiento | Solo lo que el LLM “sabe” o lo que cabe en su prompt | Documentos externos indexados |
| Tipo de embeddings usados | Internos (solo durante inferencia) | Externos (persistentes y reutilizables) |
| Búsqueda semántica | ❌ No | ✅ Sí |
| Memoria a largo plazo | ❌ Temporal | ✅ Permanente |
| Ideal para | Consultas directas, análisis tabular, ejecución de código | Preguntas sobre documentos, normas, informes |

---

#### En resumen

Un **agente clásico** *razona* sobre lo que ya tiene en su contexto.  
Un **RAG** *recuerda y busca* en su memoria externa antes de razonar.  

> Los embeddings internos permiten **comprender**,  
> mientras que los embeddings externos permiten **recordar y recuperar**.


In [10]:
import os
from getpass import getpass

# ============================================================
# Credenciales: usa Colab Secrets (nunca escribas claves en el código).
# ============================================================

def load_api_key(env_name):
    """
    Loads an API key from Colab Secrets if available.
    If it is not available, asks the user to enter it manually.
    """

    key = None

    # Try to read from Google Colab Secrets
    try:
        from google.colab import userdata
        key = userdata.get(env_name)
    except Exception:
        key = None

    # If the key is not in Secrets, ask manually
    if not key:
        key = getpass(f"Enter your {env_name}: ")

    # Store key as environment variable
    os.environ[env_name] = key

    return bool(key)


google_loaded = load_api_key("GOOGLE_API_KEY")
openai_loaded = load_api_key("OPENAI_API_KEY")

print("Entorno configurado.")
print("GOOGLE_API_KEY cargada:", google_loaded)
print("OPENAI_API_KEY cargada:", openai_loaded)

Enter your GOOGLE_API_KEY: ··········
Enter your OPENAI_API_KEY: ··········
Entorno configurado.
GOOGLE_API_KEY cargada: True
OPENAI_API_KEY cargada: False


In [11]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ruta de tu PDF (ajusta el nombre según lo subas a Colab)
PDF_PATH = "/content/2._Libro_1___Disposiciones_Generales.pdf"

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(
        f"No se encontró el archivo: {PDF_PATH}\n"
        "Sube el PDF a Colab o ajusta la variable PDF_PATH."
    )

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()   # lista de Document, normalmente uno por página
print(f"Número de páginas cargadas: {len(docs)}")

# Dividimos en chunks más pequeños para RAG
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

split_docs = text_splitter.split_documents(docs)
print(f"Número de fragmentos (chunks) después del split: {len(split_docs)}")

Número de páginas cargadas: 43
Número de fragmentos (chunks) después del split: 166


In [12]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings


def get_llm(provider: str = "gemini"):
    """Devuelve un LLM de chat según el proveedor elegido."""
    if provider == "openai":
        return ChatOpenAI(
            model="gpt-4o-mini",
            temperature=0.1,
        )

    if provider == "gemini":
        return ChatGoogleGenerativeAI(
            model="gemini-3.5-flash-lite",
            temperature=0.1,
        )

    raise ValueError("Proveedor no soportado. Usa 'openai' o 'gemini'.")


def get_embeddings(provider: str = "gemini"):
    """Devuelve el modelo de embeddings según el proveedor elegido."""
    if provider == "openai":
        return OpenAIEmbeddings(model="text-embedding-3-small")

    if provider == "gemini":
        return GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

    raise ValueError("Proveedor no soportado. Usa 'openai' o 'gemini'.")


PROVIDER = "gemini"

In [13]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


def Clasic_LLM(query, llm):
    """
    QA clásico:
    no usa recuperación vectorial; recibe directamente los fragmentos del documento.
    """
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Responde en español usando únicamente el contexto proporcionado.\n\n"
                "Contexto:\n{context}",
            ),
            ("human", "{input}"),
        ]
    )

    classic_chain = create_stuff_documents_chain(
        llm=llm,
        prompt=prompt,
    )

    # Para no exceder el contexto, puedes limitar los chunks.
    N_CHUNKS = 1000
    classic_docs = split_docs[:N_CHUNKS]

    print("========== AGENTE CLÁSICO ==========")
    classic_answer = classic_chain.invoke(
        {
            "context": classic_docs,
            "input": query,
        }
    )

    print(classic_answer)
    return classic_answer


query = "Cómo se define Campo o densidad de flujo magnético?"
llm = get_llm(PROVIDER)
Clasic_LLM(query, llm)

========== AGENTE CLÁSICO ==========
Basándose en el contexto proporcionado, el **campo o densidad de flujo magnético** se define como:

Una alteración del espacio que hace que en las cargas eléctricas en movimiento (corrientes) se genere una fuerza proporcional a su velocidad y a su carga. 

En teoría se debería hablar siempre de intensidad de campo magnético, pero en la práctica se toma la densidad de flujo magnético, la cual se representa con la letra $B$, se mide en teslas y tiene la siguiente equivalencia:

$$1 \text{ tesla} = 1 \text{ N}/(\text{A}\cdot\text{m}) = 1 \text{ V}\cdot\text{s}/\text{m}^2 = 1 \text{ Wb}/\text{m}^2 = 10.000 \text{ gauss}$$


'Basándose en el contexto proporcionado, el **campo o densidad de flujo magnético** se define como:\n\nUna alteración del espacio que hace que en las cargas eléctricas en movimiento (corrientes) se genere una fuerza proporcional a su velocidad y a su carga. \n\nEn teoría se debería hablar siempre de intensidad de campo magnético, pero en la práctica se toma la densidad de flujo magnético, la cual se representa con la letra $B$, se mide en teslas y tiene la siguiente equivalencia:\n\n$$1 \\text{ tesla} = 1 \\text{ N}/(\\text{A}\\cdot\\text{m}) = 1 \\text{ V}\\cdot\\text{s}/\\text{m}^2 = 1 \\text{ Wb}/\\text{m}^2 = 10.000 \\text{ gauss}$$'

In [ ]:
from langchain_chroma import Chroma

embeddings = get_embeddings(PROVIDER)

# Versión en memoria, no persistente.
# Si quieres persistir en disco, añade persist_directory="/content/chroma_retie".
collection_name = f"retie_capitulo_1_{PROVIDER}"

chroma_db = Chroma.from_documents(
    documents=split_docs,
    embedding=embeddings,
    collection_name=collection_name,
)

GoogleGenerativeAIError: Error embedding content: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-1.0, description: Embed content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.
Please retry in 34.658149323s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://aistudio.google.com/docs/rate-limits"
}
, violations {
  description: "Embed content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits."
  quota_metric: "generativelanguage.googleapis.com/embed_content_free_tier_requests"
  quota_id: "EmbedContentRequestsPerMinutePerUserPerProjectPerModel-

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


def RAG(query, llm):
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Responde en español usando únicamente el contexto recuperado.\n\n"
                "Contexto:\n{context}",
            ),
            ("human", "{input}"),
        ]
    )

    qa_chain = create_stuff_documents_chain(
        llm=llm,
        prompt=prompt,
    )

    rag_chain = create_retrieval_chain(
        retriever=chroma_db.as_retriever(search_kwargs={"k": 4}),
        combine_docs_chain=qa_chain,
    )

    print("========== RAG (ChromaDB) ==========")
    rag_result = rag_chain.invoke({"input": query})

    print("\n--- Respuesta RAG ---")
    print(rag_result["answer"])
    return rag_result


RAG(query, llm)

###  Evolución del RAG clásico al Agentic RAG

El **RAG clásico (Retrieval-Augmented Generation)** marcó un gran avance al permitir que un modelo de lenguaje **recupere información externa** antes de generar una respuesta.  
Sin embargo, su comportamiento sigue siendo **pasivo**: busca, concatena los fragmentos relevantes y responde, sin razonar ni decidir qué hacer a continuación.  
De ahí surge la siguiente evolución: el **Agentic RAG**.

---

####  RAG clásico — Memoria extendida con recuperación semántica

- **Función principal:** Combinar recuperación (embeddings + vector store) y generación.  
- **Flujo:**  
  1. El sistema recibe la pregunta.  
  2. Busca fragmentos relevantes en los documentos.  
  3. Los entrega al LLM para redactar una respuesta.  
- **Ventajas:**  
  - Acceso directo a fuentes externas (PDFs, normas, artículos).  
  - Respuestas contextualizadas y con evidencia textual.  
- **Limitaciones:**  
  - No decide estrategias de búsqueda.  
  - No usa herramientas externas (por ejemplo, ejecutar código, hacer cálculos o consultar bases tabulares).  
  - No mantiene diálogo razonado ni capacidad de planificación.

---

####  Agentic RAG — Razonamiento, planificación y herramientas

El **Agentic RAG** combina la base del RAG con la **inteligencia operativa de un agente**.  
Ya no es un simple sistema de recuperación, sino un **agente autónomo** capaz de:

- **Razonar paso a paso** para decidir qué información necesita.  
- **Planificar acciones** (buscar, filtrar, calcular, comparar).  
- **Usar herramientas** como:
  - Bases de datos o DataFrames (pandas agent).  
  - APIs externas (clima, normativa, mantenimiento, etc.).  
  - Módulos de código Python o motores de búsqueda.  
- **Integrar múltiples contextos** (texto, tablas, normas) dentro de una misma conversación.

---

#### Comparación conceptual

| Aspecto | RAG clásico | Agentic RAG |
|----------|--------------|-------------|
| Recuperación semántica | ✅ Sí | ✅ Sí |
| Razonamiento multi-paso | ❌ No | ✅ Sí |
| Uso de herramientas | ❌ No | ✅ Sí |
| Capacidad de planificación | ❌ No | ✅ Sí |
| Adaptación dinámica al tipo de consulta | Limitada | Alta |
| Ejemplo típico | Chat con PDF normativo | Chat con PDF + DataFrame + ejecución de código |

---

#### En resumen

- El **RAG clásico** amplía la **memoria** del modelo.  
- El **Agentic RAG** amplía su **razonamiento y autonomía**.

> *El RAG responde con evidencia.*  
> *El Agentic RAG decide cómo obtenerla, combinarla y aplicarla.*


In [ ]:
retriever = chroma_db.as_retriever(
    search_kwargs={"k": 4}
)


def buscar_en_retie(query):
    docs = retriever.invoke(query)

    partes = []
    for d in docs:
        pag = d.metadata.get("page", "NA")
        partes.append(f"[Página {pag}] {d.page_content}")

    return "\n\n".join(partes)

In [ ]:
from langchain.tools import Tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Registramos la herramienta para el agente
herramientas = [
    Tool(
        name="RETIE_RAG_BUSCADOR",
        func=buscar_en_retie,
        description=(
            "Usa esta herramienta para buscar en el documento "
            "definiciones y explicaciones técnicas. "
            "La entrada debe ser una pregunta en español."
        ),
    )
]


def AgenticRAG(query, herramientas, llm):
    """
    Agentic RAG:
    el LLM decide cuándo llamar herramientas de búsqueda.
    """
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "Eres un asistente técnico. Responde en español. "
                "Cuando necesites evidencia del documento, usa las herramientas disponibles.",
            ),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad"),
        ]
    )

    agent = create_tool_calling_agent(
        llm=llm,
        tools=herramientas,
        prompt=prompt,
    )

    agentic_rag = AgentExecutor(
        agent=agent,
        tools=herramientas,
        verbose=True,
    )

    print("========== AGENTIC RAG (con herramienta de búsqueda en RETIE) ==========")
    result = agentic_rag.invoke({"input": query})

    print("\n--- Respuesta del agente ---")
    print(result["output"])
    return result


AgenticRAG(query, herramientas, llm)

In [ ]:
from langchain_chroma import Chroma
from langchain.tools import Tool

PDF_PATH_1 = "/content/2._Libro_1___Disposiciones_Generales.pdf"
PDF_PATH_2 = "/content/4._Libro_3_-_Instalaciones.pdf"

for path in [PDF_PATH_1, PDF_PATH_2]:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No se encontró el archivo: {path}\n"
            "Sube el PDF a Colab o ajusta la ruta correspondiente."
        )

loader_1 = PyPDFLoader(PDF_PATH_1)
docs_1 = loader_1.load()

loader_2 = PyPDFLoader(PDF_PATH_2)
docs_2 = loader_2.load()

# Dividimos en chunks más pequeños para RAG
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

embeddings = get_embeddings(PROVIDER)

split_docs_cap1 = text_splitter.split_documents(docs_1)
split_docs_cap2 = text_splitter.split_documents(docs_2)

# Vector store para cada documento/capítulo
chroma_cap1 = Chroma.from_documents(
    documents=split_docs_cap1,
    embedding=embeddings,
    collection_name=f"retie_cap1_{PROVIDER}",
)

chroma_cap2 = Chroma.from_documents(
    documents=split_docs_cap2,
    embedding=embeddings,
    collection_name=f"retie_cap2_{PROVIDER}",
)

retriever1 = chroma_cap1.as_retriever(search_kwargs={"k": 4})


def buscar_en_retie_cap1(query):
    docs = retriever1.invoke(query)

    return "\n\n".join(
        f"[Cap. 1 - pág {d.metadata.get('page', 'NA')}] {d.page_content}"
        for d in docs
    )


retriever2 = chroma_cap2.as_retriever(search_kwargs={"k": 4})


def buscar_en_retie_cap2(query):
    docs = retriever2.invoke(query)

    return "\n\n".join(
        f"[Cap. 2 - pág {d.metadata.get('page', 'NA')}] {d.page_content}"
        for d in docs
    )

In [ ]:


tools = [
    Tool(
        name="BUSCAR_EN_RETIE_CAPITULO_1",
        func=buscar_en_retie_cap1,
        description=(
            "Usa esta herramienta para buscar en el CAPÍTULO 1 del RETIE, "
            "por ejemplo definiciones generales, conceptos básicos y términos fundamentales."
        ),
    ),
    Tool(
        name="BUSCAR_EN_RETIE_CAPITULO_2",
        func=buscar_en_retie_cap2,
        description=(
            "Usa esta herramienta para buscar en el CAPÍTULO 2 del RETIE, "
            "por ejemplo requisitos de instalaciones."
        ),
    ),
]
AgenticRAG(query,tools,llm)


In [ ]:
query2= "que se dice sobre Materiales reutilizados en instalaciones de uso final?"
AgenticRAG(query2,tools,llm)

In [ ]:
#import pandas as pd
from langchain_experimental.agents import create_pandas_dataframe_agent

In [ ]:
import pandas as pd

In [ ]:
# DataFrame de ejemplo. Podrías reemplazarlo por un CSV real.
df = pd.DataFrame(
    {
        "producto": ["A", "B", "C", "D"],
        "ventas": [120, 340, 230, 410],
        "costos": [60, 200, 120, 300],
    }
)

# Ejemplos de consultas en lenguaje natural
query_p_1 = "¿Cuál es el promedio de ventas y de costos?"
query_p_2 = "¿Qué producto tiene el mayor margen (ventas - costos) y cuánto es?"


def AgenticPandas(query, df, llm):
    agent = create_pandas_dataframe_agent(
        llm,
        df,
        verbose=True,
        agent_type="tool-calling",
        allow_dangerous_code=True,
        include_df_in_prompt=True,
        number_of_head_rows=5,
    )

    result = agent.invoke({"input": query})
    print(result["output"])
    return result


AgenticPandas(query_p_1, df, llm)

In [ ]:
AgenticPandas(query_p_2,df,llm)

In [ ]:
query_plot = "Construye un gráfico en el cual muestres las ventas por productos"
path_plot = "/content/image.jpg"


def AgenticPandasPlot(query, path, df, llm):
    prompt = f"""
    {query}.

    Construye el gráfico de la forma más estética posible para mostrar a un usuario.
    Puedes utilizar colores verdes y grises en diferentes tonalidades si es necesario.
    Los títulos y ejes deben estar en español.
    Guarda la imagen en la ruta: {path}
    No ejecutes plt.show().
    Siempre ejecuta plt.tight_layout() antes de guardar.
    """

    agent = create_pandas_dataframe_agent(
        llm,
        df,
        verbose=True,
        agent_type="tool-calling",
        allow_dangerous_code=True,
        include_df_in_prompt=True,
    )

    response = agent.invoke({"input": prompt})
    print(response["output"])
    return response


AgenticPandasPlot(query_plot, path_plot, df, llm)

# Implementación local

In [ ]:
# colab-xterm se instaló en la celda inicial.
%load_ext colabxterm

## **Códigos a ejecutar en la línea de comandos**

Las líneas comentadas en la siguiente celda son **comandos de terminal**: debes ejecutarlas manualmente dentro de la ventana `%xterm` (sin el símbolo `#`). `langchain-ollama` no descarga ni inicia Ollama por sí solo. Mantén abierto `ollama serve` mientras usas los modelos locales.

In [ ]:
%xterm
# apt-get update
# apt-get install -y zstd
# curl -fsSL https://ollama.com/install.sh | sh
# ollama serve

## Instalación de modelos

In [ ]:
%xterm
# ollama pull llama3.2:1b
# ollama pull qwen2.5:1.5b
# ollama pull deepseek-r1:1.5b

In [ ]:
from langchain_ollama import ChatOllama

OLLAMA_MODEL = "llama3.2:1b"   # o "qwen2.5:1.5b", "deepseek-r1:1.5b"


def get_ollama_llm(model_name: str = OLLAMA_MODEL):
    """
    Devuelve un ChatOllama listo para usarse en:
    - QA clásico
    - RAG
    - Agentic RAG, si el modelo local soporta tool calling
    - Agente de Pandas, si el modelo local soporta tool calling
    """
    return ChatOllama(
        model=model_name,
        temperature=0.1,
    )


# LLM principal para estos ejercicios con Ollama
llm_ollama = get_ollama_llm()
print(f"Usando modelo de Ollama: {OLLAMA_MODEL}")

In [ ]:
Clasic_LLM(query,llm_ollama)

In [ ]:
RAG(query,llm_ollama)

In [ ]:
AgenticRAG(query,herramientas,llm_ollama)

In [ ]:
AgenticRAG(query2,tools,llm_ollama)

In [ ]:
AgenticPandas(query_p_1,df,llm_ollama)

In [ ]:
AgenticPandas(query_p_2,df,llm_ollama)

In [ ]:
AgenticPandasPlot(query_plot,path_plot,df,llm_ollama)

In [ ]:
import matplotlib.pyplot as plt

# Muestra el dataframe original
print(df.head())

# Crea un gráfico de barras para mostrar las ventas por producto
plt.figure(figsize=(10, 6))
bars = plt.bar(df["producto"], df["ventas"])

# Agrega un título al gráfico
plt.title("Ventas por producto")

# Agrega etiquetas a las barras
for bar, producto, ventas in zip(bars, df["producto"], df["ventas"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        ventas + 10,
        f"{producto}: {ventas}",
        ha="center",
        va="bottom",
    )

# Añade etiquetas a los ejes
plt.xlabel("Producto")
plt.ylabel("Ventas (unidades)")

# Ajusta el tamaño del gráfico y guarda la imagen
plt.tight_layout()
plt.savefig("/content/image.jpg", dpi=300)
plt.show()